# Use posterior draws

Extract draws, compute predictions, and save a finished run.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

## Prepare the example

These examples require the regression `model` and sampling `results` from
{doc}`tutorials/md/01c-transform`, with `sigma_sq` included in the stored draws.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


In [3]:
results = gs.LieselMCMC(model).run_for_epochs(
    seed=1,
    num_chains=4,
    adaptation=1000,
    posterior=1000,
    positions_included=["sigma_sq"],
    show_progress=False,
)

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


## Extract the posterior

In [4]:
samples = results.get_posterior_samples()
sample_shapes = pd.DataFrame(
    {
        "Variable": ["beta", "sigma_sq"],
        "Shape": [samples["beta"].shape, samples["sigma_sq"].shape],
    },
)

In [5]:
sample_shapes

,Variable,Shape
0,beta,"(4, 1000, 2)"
1,sigma_sq,"(4, 1000)"


The shapes are `(4, 1000, 2)` and `(4, 1000)` in the tutorial:
chains, draws, then any parameter dimensions. Preserve the first two axes
when computing diagnostics. {meth}`get_samples() <liesel.goose.SamplingResults.get_samples>` also includes initial values
and warmup; use {meth}`get_posterior_samples() <liesel.goose.SamplingResults.get_posterior_samples>` for posterior inference.

## Record derived values

Goose records variables updated by kernels. The tutorial also sets
`positions_included=["sigma_sq"]` to record the derived variance.

Here NUTS samples `log_sigma_sq`; `sigma_sq` is its positive transformation.
`positions_excluded` overrides included positions and affects storage only.
It does not remove a kernel. Excluding sampled parameters can prevent later
predictions or diagnostics that need those draws.

For large intermediate arrays, retain the underlying parameter draws and
recompute quantities as needed instead:

In [6]:
variance = model.predict(samples, predict=["sigma_sq"])

In [7]:
gs.SamplesSummary(variance).to_dataframe()[["mean", "sd"]].round(3)

,mean,sd
variable,,
sigma_sq,1.039,0.065


## Predict new responses

For the tutorial's design matrix variable `X`, construct a grid with the
same two columns: intercept and covariate.

In [8]:
x_grid = jnp.linspace(0.0, 1.0, 50)
X_grid = jnp.column_stack([jnp.ones_like(x_grid), x_grid])

mean = model.predict(
    samples,
    predict=["mu"],
    newdata={"X": X_grid},
)
mean_summary = gs.SamplesSummary(mean, which=["quantiles"]).to_dataframe()
mean_summary["x"] = x_grid

In [9]:
mean_summary.iloc[[0, -1]][["x", "q_0.05", "q_0.5", "q_0.95"]].round(3)

,x,q_0.05,q_0.5,q_0.95
variable,,,,
mu,0.0,0.839,0.992,1.140
mu,1.0,2.741,2.888,3.041


`mean["mu"]` contains the regression mean at each grid point for every draw.
The displayed endpoints show how the fitted mean rises across the grid.
The 5th and 95th percentiles form a pointwise 90% credible interval for
the mean, as plotted in the first tutorial.

To include observation noise, simulate responses conditional on those draws:

In [10]:
parameter_draws = {name: samples[name] for name in model.parameters}
predictive = model.sample(
    sample_shape=(),
    seed=jax.random.key(2027),
    posterior_samples=parameter_draws,
    newdata={"X": X_grid, "y": jnp.zeros_like(x_grid)},
)
y_predictive = predictive["y"]

In [11]:
y_predictive.shape

(4, 1000, 50)

Pass draws for the writable parameters to {meth}`model.sample <liesel.model.Model.sample>`. The recorded
`sigma_sq` is a derived value in this model; it is recomputed from
`log_sigma_sq` rather than set directly.

The placeholder `y` gives the response the new grid length while the model
updates its state. These values are replaced by simulated responses; they are
not additional observations used for fitting.

With `shape=()`, there is one simulated response vector per posterior draw.
Predictive intervals also include observation variation and are generally
wider. See {meth}`~liesel.model.Model.predict` and
{meth}`~liesel.model.Model.sample` for new-data and shape rules. Additional
sample dimensions, if requested, precede the posterior chain/draw dimensions.

## Save a finished run

In [12]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "regression-samples.pkl"
    results.pkl_save(path)
    restored = gs.SamplingResults.pkl_load(path)

This example uses a temporary directory; choose a persistent path to keep your
run. Load only trusted pickle files. Keep the model definition, data, and
{doc}`environment information <tutorials/md/05-reproducibility>` with the result.

Alternatively, pass `save_path="regression-samples.pkl"` to
{meth}`run_for_epochs <liesel.goose.LieselMCMC.run_for_epochs>`. If that path already exists, Goose **loads it and skips
sampling**, even when you changed the model, seed, or sampling settings.
Use a fresh path for a new fit. A saved {class}`SamplingResults <liesel.goose.SamplingResults>` object contains
results; it is not an engine checkpoint for resuming a chain.